# PlanLiftAR geometry v2 — rooms-first pipeline
Plan: `planliftar-v1.md`. Layers: hysteresis healing -> micro-gap sealing -> virtual door seals -> rooms-first extraction -> consensus + invariants -> barrier.
Phase 1 = Cells CFG-6. Cells 7-10 stubbed for Phase 2.

In [ ]:
# CFG — the only tunables in the pipeline (plan: single config cell)
HYST_LOW       = 0.25   # hysteresis weak threshold; sweep 0.15-0.35
HYST_HIGH      = 0.50
SEAL_FACTOR    = 1.5    # seal openings narrower than SEAL_FACTOR * wall thickness
CONSENSUS_MIN  = 0.70   # Layer 3 threshold (Phase 2)
MIN_ROOM_AREA  = 2000   # px^2, same default as v1 spike
WALL_HEIGHT    = 2.5    # relative units for GLB extrusion (Phase 3)

In [ ]:
# Cell 0 — environment and weights (unchanged from mitunet-wall-test, + scikit-image)
!pip install -q segmentation-models-pytorch==0.5.0 albumentations opencv-python-headless scikit-image

import torch
print("torch", torch.__version__, "| CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

WEIGHTS = "MitUNet_cubicasa-5k_a62_mit_b4_tversky_7666_30E.pth"
URL = f"https://media.githubusercontent.com/media/aliasstudio/mitunet/master/experiments/models/{WEIGHTS}"
import os
if not os.path.exists(WEIGHTS):
    !wget -q -O {WEIGHTS} "{URL}"
print(WEIGHTS, "->", round(os.path.getsize(WEIGHTS)/1e6, 1), "MB (expect ~257)")

In [ ]:
# Cell 1 — model assembly and load (unchanged)
import segmentation_models_pytorch as smp
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

aux_segformer = smp.Segformer(encoder_name="mit_b4", encoder_weights=None)
model = smp.Unet(encoder_name="mit_b4", encoder_weights=None,
                 in_channels=3, classes=1, decoder_attention_type="scse")
model.encoder = aux_segformer.encoder
model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE, weights_only=True))
model.to(DEVICE).eval()
print("MitUNet loaded")

In [ ]:
# Cell 2 — inference with auto-crop second pass (keeps probability map)
import cv2, numpy as np, albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

IMAGE_PATH = "/kaggle/input/datasets/qmarva/cubicasa5k/cubicasa5k/cubicasa5k/colorful/10052/F1_original.png"
image_rgb_full = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)
Hf, Wf = image_rgb_full.shape[:2]

transform = A.Compose([A.Resize(512, 512),
                       A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
                       ToTensorV2()])

def infer(img):
    inp = transform(image=img)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        p = torch.sigmoid(model(inp)).squeeze().cpu().numpy()
    return cv2.resize(p, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_LINEAR)

import os
CACHED = "/kaggle/input/wall-prob-cache/wall_prob_crop.npy"
if os.path.exists(CACHED):
    prob_full = np.load(CACHED)
    cx0, cy0 = [int(v) for v in open("/kaggle/input/wall-prob-cache/crop_offset.txt").read().split(",")]
    image_rgb = image_rgb_full[cy0:cy0+prob_full.shape[0], cx0:cx0+prob_full.shape[1]]
    print("using cached crop prob map")
else:
    # pass 1: whole sheet, find plan extent (plan may be a small region of a big sheet)
    prob1 = infer(image_rgb_full)
    ys, xs = np.nonzero(prob1 > 0.3)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    mw, mh = int(0.08*(x1-x0)), int(0.08*(y1-y0))
    cx0, cy0 = max(0, x0-mw), max(0, y0-mh)
    cx1, cy1 = min(Wf, x1+mw), min(Hf, y1+mh)
    if (cx1-cx0)*(cy1-cy0) < 0.5 * Hf * Wf:      # plan is a sub-region: re-infer on crop
        image_rgb = image_rgb_full[cy0:cy1, cx0:cx1]
        prob_full = infer(image_rgb)
        cx0, cy0 = int(cx0), int(cy0)
        print(f"auto-crop: x[{cx0}:{cx1}] y[{cy0}:{cy1}]")
    else:
        image_rgb, prob_full, (cx0, cy0) = image_rgb_full, prob1, (0, 0)
        print("plan fills sheet; no crop")
H, W = prob_full.shape
np.save("/kaggle/working/wall_prob_crop.npy", prob_full)
open("/kaggle/working/crop_offset.txt", "w").write(f"{cx0},{cy0}")
print("prob map:", prob_full.shape, "offset", (cx0, cy0))

In [ ]:
# Cell 3 — Layer 2: hysteresis thresholding (heals confidence-dip breaks)
from skimage.filters import apply_hysteresis_threshold

mask_naive = (prob_full > HYST_HIGH).astype(np.uint8)
mask = apply_hysteresis_threshold(prob_full, low=HYST_LOW, high=HYST_HIGH).astype(np.uint8)

def n_comp(m):
    n, _ = cv2.connectedComponents(m); return n - 1
print(f"components: naive@0.5 = {n_comp(mask_naive)}  ->  hysteresis = {n_comp(mask)}   (healing metric; ideal ~1)")

In [ ]:
# Cell 4 — Layer 1: wall thickness estimate + micro-gap sealing (relative units, no fixed pixels)
from skimage.morphology import skeletonize

dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
skel = skeletonize(mask.astype(bool))
t_med = 2 * float(np.median(dist[skel]))          # median wall thickness, px
SEAL_BELOW = SEAL_FACTOR * t_med                  # breaks are hairline; doors ~3x thickness+
k = int(SEAL_BELOW) | 1
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
print(f"t_med = {t_med:.1f}px  seal kernel = {k}px  components after close = {cv2.connectedComponents(mask)[0]-1}")
# risk note (plan): CLOSE can bridge corridors narrower than kernel — if component
# count drops implausibly, verify seals against prob_full before trusting the mask

In [ ]:
# Cell 5 — sealing hints for labeling only (never exported):
# union CubiCasa-detected walls + width-adaptive bars over its door/window openings.
# MitUNet stays the geometry source; these seals exist so rooms label as closed pockets.
import json, os, io

def find_file(name):
    for base in ("/kaggle/working", "/kaggle/input"):
        for dp, _, fs in os.walk(base):
            if name in fs: return os.path.join(dp, name)
    raise FileNotFoundError(name)

sg = json.load(open(find_file("sample_geometry.json")))            # CubiCasa walls + openings
rf = json.load(open(find_file("cubicasa_rooms_fixtures.json")))    # CubiCasa rooms (Phase 2 consensus)
cubicasa_rooms = rf.get("rooms", [])
fixtures = rf.get("fixtures", [])

walls_np = [(np.array(w["start"], float), np.array(w["end"], float),
             w.get("thickness_px", t_med)) for w in sg["walls"]]

cubi = np.zeros_like(mask)
for a, b, th in walls_np:                     # shift sheet coords into crop frame
    cv2.line(cubi, (int(a[0]-cx0), int(a[1]-cy0)), (int(b[0]-cx0), int(b[1]-cy0)),
             1, thickness=int(max(2, th)))

def nearest_wall_dir(p):
    best, bd = None, 1e18
    for a, b, th in walls_np:
        ab = b - a; L2 = ab @ ab
        t = 0 if L2 == 0 else np.clip((p - a) @ ab / L2, 0, 1)
        d2 = ((a + t*ab) - p) @ ((a + t*ab) - p)
        if d2 < bd: bd, best = d2, ab
    nn = np.linalg.norm(best)
    return (best / nn) if nn else np.array([1.0, 0.0])

seal_mask = cv2.bitwise_or(mask, cubi)
sealed = 0
for o in sg["openings"]:                      # doors AND windows both seal for labeling
    p = np.array(o["position"], float)
    fx, fy = int(p[0]-cx0), int(p[1]-cy0)
    if not (0 <= fx < W and 0 <= fy < H): continue
    d = nearest_wall_dir(p)
    L = 1.6 * o["width_px"]                   # actual opening width + margin
    cv2.line(seal_mask, (int(fx-d[0]*L/2), int(fy-d[1]*L/2)),
             (int(fx+d[0]*L/2), int(fy+d[1]*L/2)), 1, thickness=max(6, int(t_med)))
    sealed += 1
print(f"seal layer: {int(cubi.sum())} CubiCasa wall px + {sealed} opening bars (labeling copy only)")

In [ ]:
# Cell 6 — rooms-first region extraction + nearest-label wall derivation
from scipy import ndimage

n, labels, stats, cents = cv2.connectedComponentsWithStats(
        cv2.bitwise_not(seal_mask * 255), connectivity=4)

Himg, Wimg = labels.shape
exterior = {l for l in range(1, n)
            if stats[l, cv2.CC_STAT_LEFT] == 0 or stats[l, cv2.CC_STAT_TOP] == 0
            or stats[l, cv2.CC_STAT_LEFT] + stats[l, cv2.CC_STAT_WIDTH] >= Wimg
            or stats[l, cv2.CC_STAT_TOP] + stats[l, cv2.CC_STAT_HEIGHT] >= Himg}
room_ids = [l for l in range(1, n)
            if l not in exterior and stats[l, cv2.CC_STAT_AREA] >= MIN_ROOM_AREA]
print(f"labels: {n-1}  exterior: {len(exterior)}  rooms: {len(room_ids)}")

rooms = []
for l in room_ids:
    rm = (labels == l).astype(np.uint8)
    cnts, _ = cv2.findContours(rm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)
    poly = cv2.approxPolyDP(c, 0.01 * cv2.arcLength(c, True), True)[:, 0, :]
    rooms.append({"label": int(l), "polygon": [{"x": int(x), "y": int(y)} for x, y in poly],
                  "area_px": int(stats[l, cv2.CC_STAT_AREA])})
print("room polygons (closed by construction):", [len(r["polygon"]) for r in rooms], "vertices")

# --- wall derivation: assign every wall pixel to nearest region label ---
# (1-px dilate-and-intersect does NOT work: regions sit a full wall thickness apart)
_, (iy, ix) = ndimage.distance_transform_edt(labels == 0, return_indices=True)
assign = labels[iy, ix]                          # every wall pixel -> nearest region label

wall_px = seal_mask.astype(bool)
a = np.where(wall_px, assign, 0)
pairs = {}
for shift, axis in ((1, 1), (1, 0)):
    b = np.roll(a, shift, axis=axis)
    edge = wall_px & np.roll(wall_px, shift, axis=axis) & (a != b) & (a > 0) & (b > 0)
    ys, xs = np.nonzero(edge)
    for y, x in zip(ys, xs):
        key = tuple(sorted((int(a[y, x]), int(b[y, x]))))
        pairs.setdefault(key, []).append((x, y))

walls = []
wid = 0
for (la, lb), pts in pairs.items():
    if len(pts) < 5: continue
    pm = np.zeros_like(seal_mask)
    for x, y in pts: pm[y, x] = 1
    cnts, _ = cv2.findContours(pm, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    for c in cnts:
        approx = cv2.approxPolyDP(c, 3, False)[:, 0, :]
        for i in range(len(approx) - 1):
            (x1, y1), (x2, y2) = approx[i], approx[i+1]
            if (int(x1)-int(x2))**2 + (int(y1)-int(y2))**2 < 25: continue
            mx, my = int((x1+x2)//2), int((y1+y2)//2)
            walls.append({"id": f"w{wid}",
                          "start": {"x": int(x1), "y": int(y1)}, "end": {"x": int(x2), "y": int(y2)},
                          "thickness": round(float(2*dist[np.clip(my,0,Himg-1), np.clip(mx,0,Wimg-1)]), 1) or round(t_med,1),
                          "between": [int(la), int(lb)],
                          "exterior": bool(la in exterior or lb in exterior)})
            wid += 1
print(f"derived {len(walls)} wall runs from {len(pairs)} adjacent label pairs")

# Phase 1 gate visual
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
axes[0].imshow(mask, cmap="gray"); axes[0].set_title("healed mask")
axes[1].imshow(np.where(np.isin(labels, room_ids), labels, 0), cmap="tab20"); axes[1].set_title(f"{len(rooms)} rooms")
axes[2].imshow(image_rgb)
for w in walls:
    axes[2].plot([w["start"]["x"], w["end"]["x"]], [w["start"]["y"], w["end"]["y"]], "r-", lw=2)
axes[2].set_title(f"{len(walls)} derived walls")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/phase1_gate.png", dpi=100); plt.show()

In [ ]:
# Cell 7-pre — consensus helpers (reviewed deliverable)
import numpy as np
import cv2
from scipy.optimize import linear_sum_assignment


def rasterize_rooms(cubicasa_rooms, shape, cx0, cy0):
    """Shift each CubiCasa room polygon into crop coords and rasterize to a filled mask, skipping ones fully outside the crop."""
    h, w = shape
    out = []
    for room in cubicasa_rooms:
        poly = room.get("polygon", [])
        if len(poly) < 3:
            continue
        pts = np.array([[p["x"] - cx0, p["y"] - cy0] for p in poly], dtype=np.float64)
        x0, y0 = pts[:, 0].min(), pts[:, 1].min()
        x1, y1 = pts[:, 0].max(), pts[:, 1].max()
        if x1 < 0 or y1 < 0 or x0 >= w or y0 >= h:
            continue
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [pts.round().astype(np.int32)], 1)
        out.append((room, mask))
    return out


def pairwise_iou(labels, room_ids, cubi_masks):
    """Compute the IoU matrix between each label-map room region and each rasterized CubiCasa mask."""
    n, m = len(room_ids), len(cubi_masks)
    iou = np.zeros((n, m), dtype=np.float64)
    for i, rid in enumerate(room_ids):
        rmask = labels == rid
        for j, cmask in enumerate(cubi_masks):
            cm = cmask.astype(bool)
            inter = np.count_nonzero(rmask & cm)
            union = np.count_nonzero(rmask | cm)
            iou[i, j] = inter / union if union else 0.0
    return iou


def hungarian_match(iou_matrix, min_iou=0.5):
    """Solve the IoU-maximizing 1:1 assignment via the Hungarian algorithm, keeping only matches >= min_iou."""
    if iou_matrix.size == 0:
        return []
    row_idx, col_idx = linear_sum_assignment(-iou_matrix)
    matches = []
    for i, j in zip(row_idx, col_idx):
        v = float(iou_matrix[i, j])
        if v >= min_iou:
            matches.append((int(i), int(j), v))
    return matches


def shoelace_area(poly):
    """Absolute polygon area via the shoelace formula for a list of {"x","y"} dict vertices."""
    xs = np.array([p["x"] for p in poly], dtype=np.float64)
    ys = np.array([p["y"] for p in poly], dtype=np.float64)
    return float(0.5 * abs(np.dot(xs, np.roll(ys, -1)) - np.dot(ys, np.roll(xs, -1))))


def polygon_is_closed(poly):
    """True if poly has >=3 vertices and encloses nonzero (shoelace) area, i.e. isn't degenerate."""
    if len(poly) < 3:
        return False
    return shoelace_area(poly) > 0.0


def _point_seg_dist(p, a, b):
    """Euclidean distance from point p to segment a-b (each an (x, y) float array)."""
    ap = p - a
    ab = b - a
    denom = np.dot(ab, ab)
    t = 0.0 if denom == 0 else np.clip(np.dot(ap, ab) / denom, 0.0, 1.0)
    proj = a + t * ab
    return float(np.linalg.norm(p - proj))


def wall_endpoint_degrees(walls, tol=6.0):
    """For each wall's start/end, count other walls (by endpoint or segment-body proximity) within tol px."""
    n = len(walls)
    pts = [(np.array([w["start"]["x"], w["start"]["y"]], dtype=np.float64),
            np.array([w["end"]["x"], w["end"]["y"]], dtype=np.float64)) for w in walls]
    degrees = {}
    for i, w in enumerate(walls):
        s_i, e_i = pts[i]
        deg_s = 0
        deg_e = 0
        for j in range(n):
            if j == i:
                continue
            s_j, e_j = pts[j]
            if np.linalg.norm(s_i - s_j) <= tol or np.linalg.norm(s_i - e_j) <= tol:
                deg_s += 1
            elif _point_seg_dist(s_i, s_j, e_j) <= tol:
                deg_s += 1
            if np.linalg.norm(e_i - s_j) <= tol or np.linalg.norm(e_i - e_j) <= tol:
                deg_e += 1
            elif _point_seg_dist(e_i, s_j, e_j) <= tol:
                deg_e += 1
        degrees[w["id"]] = (deg_s, deg_e)
    return degrees


def build_room_masks(labels, room_ids):
    """Map each room label id to a uint8 binary mask of its label-map region."""
    return {rid: (labels == rid).astype(np.uint8) for rid in room_ids}


In [ ]:
# Cell 7 — Layer 3: consensus scoring + invariants + routing
cubi_pairs = rasterize_rooms(cubicasa_rooms, labels.shape, cx0, cy0)
# keep only rooms mostly inside the crop — neighbors' partial rooms inflate the denominator
cubi_pairs = [(r, m) for r, m in cubi_pairs
              if m.sum() >= 0.5 * max(shoelace_area(r["polygon"]), 1)]
# and only claims whose centroid lands inside a MitUNet room region — neighbor
# slivers and corridor fragments land on exterior/wall pixels and drop out
def _centroid_label(m):
    ys, xs = np.nonzero(m)
    return labels[int(ys.mean()), int(xs.mean())] if len(xs) else 0
cubi_pairs = [(r, m, _centroid_label(m)) for r, m in cubi_pairs]
cubi_pairs = [(r, m, l) for r, m, l in cubi_pairs if l in room_ids]
cubi_masks = [m for _, m, _ in cubi_pairs]
room_masks = build_room_masks(labels, room_ids)

if room_ids and cubi_masks:
    iou = pairwise_iou(labels, room_ids, cubi_masks)
    matches = hungarian_match(iou, min_iou=0.5)
else:
    iou, matches = np.zeros((0, 0)), []
consensus = len(matches) / max(len(room_ids), len(cubi_masks), 1)
print(f"consensus: {len(matches)} matches / max({len(room_ids)} mitunet, {len(cubi_masks)} cubicasa) = {consensus:.2f}")

# self-contained invariants (no second source needed)
inv = {
    "all_polygons_closed": all(polygon_is_closed(r["polygon"]) for r in rooms),
    "min_area_ok":         all(r["area_px"] >= MIN_ROOM_AREA for r in rooms),
    "room_count_sane":     1 <= len(rooms) <= 40,
    "has_rooms":           len(rooms) > 0,
}
degs = wall_endpoint_degrees(walls)
n_loose = sum((d[0] < 1) + (d[1] < 1) for d in degs.values())
inv["loose_endpoints_warn"] = n_loose        # warning only: half walls are legitimate
print("invariants:", inv)

# routing (product decision): count mismatch alone NEVER triggers repair — an
# open-plan merged region that passes invariants is accepted with a note.
# Repair (Cell 8) fires only when low consensus coincides with a geometry anomaly.
hard_fail = not (inv["all_polygons_closed"] and inv["min_area_ok"] and inv["has_rooms"] and inv["room_count_sane"])
merged_rooms_note = consensus < CONSENSUS_MIN and not hard_fail
needs_repair = consensus < CONSENSUS_MIN and hard_fail
print(f"routing: hard_fail={hard_fail}  merged_rooms_note={merged_rooms_note}  needs_repair={needs_repair}")

In [ ]:
# Cell 8 — Layer 4: watershed VIRTUAL split for consensus scoring.
# Locked decision: the pipeline never invents walls — an accepted merged open-plan
# region exports as one room. The split below exists only to score how well the
# merged regions decompose into the sub-rooms CubiCasa claims (semantic consensus).
from skimage.segmentation import watershed as sk_watershed

consensus_split = consensus
if merged_rooms_note and cubi_pairs:
    vlabels = np.zeros_like(labels); nxt = 1
    for rid in room_ids:
        region = labels == rid
        seeds = [np.nonzero(m) for r, m, l in cubi_pairs if l == rid]
        seeds = [(int(ys.mean()), int(xs.mean())) for ys, xs in seeds if len(xs)]
        if len(seeds) <= 1:
            vlabels[region] = nxt; nxt += 1; continue
        markers = np.zeros_like(labels)
        for k, c in enumerate(seeds, 1): markers[c] = k
        d = cv2.distanceTransform(region.astype(np.uint8), cv2.DIST_L2, 5)
        ws = sk_watershed(-d, markers=markers, mask=region)
        for k in range(1, len(seeds) + 1):
            if (ws == k).any(): vlabels[ws == k] = nxt; nxt += 1
    vroom_ids = [v for v in range(1, nxt)]
    iou2 = pairwise_iou(vlabels, vroom_ids, cubi_masks)
    matches2 = hungarian_match(iou2, min_iou=0.5)
    consensus_split = len(matches2) / max(len(vroom_ids), len(cubi_masks), 1)
    print(f"virtual split: {len(room_ids)} regions -> {len(vroom_ids)} sub-rooms; "
          f"consensus {consensus:.2f} -> {consensus_split:.2f}")
else:
    print("no split needed")
consensus_final = max(consensus, consensus_split)

In [ ]:
# Cell 9 — Layer 5: acceptance barrier (single decision point, machine-readable)
reasons = []
if not inv["has_rooms"]:            reasons.append("no_rooms")
if not inv["all_polygons_closed"]:  reasons.append("open_polygon")
if not inv["min_area_ok"]:          reasons.append("room_below_min_area")
if not inv["room_count_sane"]:      reasons.append("room_count_implausible")
if needs_repair:                    reasons.append("low_consensus_with_geometry_anomaly")

passed = not reasons
report = {
  "accepted": bool(passed),
  "reject_reasons": reasons,
  "notes": (["merged_rooms"] if merged_rooms_note else []),
  "metrics": {"rooms_mitunet": len(rooms), "rooms_cubicasa": len(cubi_masks),
              "consensus_iou": round(consensus_final, 3), "consensus_raw": round(consensus, 3), "matches": len(matches),
              "wall_thickness_px": round(t_med, 1), "loose_endpoints": int(n_loose),
              "crop_offset": [int(cx0), int(cy0)]}
}
json.dump(report, open("/kaggle/working/acceptance_report.json", "w"), indent=2)
print(json.dumps(report, indent=2))

In [ ]:
# Cell 10 — export (door spans carved out) + QA. A rejected plan writes NO geometry file.
def carve_doors(walls_in, openings, off_x, off_y, t):
    """Split wall runs where a door opening crosses them; windows stay (they are wall in 3D)."""
    out = []
    for w in walls_in:
        a = np.array([w["start"]["x"], w["start"]["y"]], float)
        b = np.array([w["end"]["x"], w["end"]["y"]], float)
        segs = [(a, b)]
        for o in openings:
            if o["type"] != "door": continue
            p = np.array(o["position"], float) - [off_x, off_y]
            half = 0.5 * o["width_px"]
            nxt = []
            for (s, e) in segs:
                d = e - s; L = float(np.linalg.norm(d))
                if L < 1e-6: continue
                u = d / L
                tproj = float(np.clip((p - s) @ u, 0, L))
                foot = s + tproj * u
                if np.linalg.norm(foot - p) > max(t, half):   # door not on this run
                    nxt.append((s, e)); continue
                lo, hi = max(0.0, tproj - half), min(L, tproj + half)
                if lo > 4:      nxt.append((s, s + lo * u))
                if L - hi > 4:  nxt.append((s + hi * u, e))
            segs = nxt
        for i, (s, e) in enumerate(segs):
            out.append({**w,
                "id": f'{w["id"]}_{i}' if len(segs) > 1 else w["id"],
                "start": {"x": round(float(s[0]), 1), "y": round(float(s[1]), 1)},
                "end":   {"x": round(float(e[0]), 1), "y": round(float(e[1]), 1)}})
    return out

if report["accepted"]:
    walls_carved = carve_doors(walls, sg["openings"], cx0, cy0, t_med)
    def to_sheet(pt): return {"x": pt["x"] + cx0, "y": pt["y"] + cy0}
    export = {
      "schema_version": "2.0",
      "source": {"image_width_px": int(Wf), "image_height_px": int(Hf)},
      "scale": {"units": "relative_px", "user_scale_required": True},
      "walls": [{**w, "start": to_sheet(w["start"]), "end": to_sheet(w["end"])} for w in walls_carved],
      "rooms": [{"id": f'room_{r["label"]}',
                 "polygon": [{"x": p["x"] + cx0, "y": p["y"] + cy0} for p in r["polygon"]],
                 "area_px": r["area_px"]} for r in rooms],
      "fixtures": fixtures,
      "acceptance": report["metrics"],
    }
    json.dump(export, open("/kaggle/working/sample_response.json", "w"), indent=1)
    print(f"exported {len(walls_carved)} wall segs ({len(walls)} pre-carve), {len(rooms)} rooms")
else:
    walls_carved = []
    print("REJECTED — no geometry file written:", report["reject_reasons"])

# QA figure
fig, axes = plt.subplots(1, 3, figsize=(21, 8))
axes[0].imshow(mask, cmap="gray"); axes[0].set_title("healed mask")
axes[1].imshow(np.where(np.isin(labels, room_ids), labels, 0), cmap="tab20")
axes[1].set_title(f'{len(rooms)} rooms | consensus {consensus:.2f}')
axes[2].imshow(image_rgb)
for w in walls_carved:
    axes[2].plot([w["start"]["x"] - cx0, w["end"]["x"] - cx0],
                 [w["start"]["y"] - cy0, w["end"]["y"] - cy0], "r-", lw=2)
axes[2].set_title(f'{len(walls_carved)} exported walls (doors carved)')
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/phase2_qa.png", dpi=100); plt.show()

In [ ]:
# Cell 11-pre — GLB builder (reviewed deliverable)
!pip install -q trimesh
# Notebook does: !pip install -q trimesh   (run that in the cell above this one)
import numpy as np
import trimesh


def _px_to_m(value, pixels_per_meter):
    """Convert a px measurement to meters given a pixels_per_meter scale factor."""
    return float(value) / float(pixels_per_meter)


def _wall_box(start_xy_m, end_xy_m, thickness_m, height_m, name):
    """Build a box mesh spanning start->end (plan-space meters), sitting on y=0.

    Plan (x, y) -> world (x, z), with z flipped (world_z = -plan_y) so the
    plan reads correctly (not mirrored) when viewed from above in a
    glTF y-up viewer. Returns None for a degenerate (zero-length) wall.
    """
    sx, sz = start_xy_m[0], -start_xy_m[1]
    ex, ez = end_xy_m[0], -end_xy_m[1]

    dx, dz = ex - sx, ez - sz
    length = float(np.hypot(dx, dz))
    if length < 1e-6:
        return None

    ux = np.array([dx, 0.0, dz]) / length          # wall's long axis, world XZ
    uz_world = np.array([0.0, 1.0, 0.0])            # world up
    uy = np.cross(uz_world, ux)                     # wall's thickness axis (XZ plane)
    uy = uy / np.linalg.norm(uy)

    box = trimesh.creation.box(extents=[length, thickness_m, height_m])

    transform = np.eye(4)
    transform[:3, 0] = ux
    transform[:3, 1] = uy
    transform[:3, 2] = uz_world
    transform[:3, 3] = [
        (sx + ex) / 2.0,
        height_m / 2.0,
        (sz + ez) / 2.0,
    ]
    box.apply_transform(transform)

    box.visual = trimesh.visual.texture.TextureVisuals(
        material=trimesh.visual.material.PBRMaterial(
            baseColorFactor=[235, 232, 225, 255],
            metallicFactor=0.0,
            roughnessFactor=0.9,
        )
    )
    box.metadata["name"] = name
    return box


def build_glb(geometry: dict, pixels_per_meter: float, wall_height_m: float = 2.5,
              out_path: str = "/kaggle/working/planlift.glb"):
    """Build a GLB from exported floorplan geometry: one box mesh per wall plus a floor slab.

    geometry["walls"] entries look like:
        {"id": ..., "start": {"x", "y"}, "end": {"x", "y"}, "thickness": <m>, "height": <m>}
    "start"/"end" are in px and are converted to meters via pixels_per_meter.
    "thickness" (and any per-wall "height") in the sample schema are already in
    meters, so they are used as-is, with wall_height_m as the fallback height
    when a wall has no "height" field.
    """
    walls = geometry.get("walls", [])

    wall_meshes = []
    names = []
    for i, wall in enumerate(walls):
        start_m = (
            _px_to_m(wall["start"]["x"], pixels_per_meter),
            _px_to_m(wall["start"]["y"], pixels_per_meter),
        )
        end_m = (
            _px_to_m(wall["end"]["x"], pixels_per_meter),
            _px_to_m(wall["end"]["y"], pixels_per_meter),
        )
        thickness_m = float(wall.get("thickness", 0.1))
        height_m = float(wall.get("height", wall_height_m))
        name = f"w{i}"

        mesh = _wall_box(start_m, end_m, thickness_m, height_m, name)
        if mesh is None:
            continue
        wall_meshes.append(mesh)
        names.append(name)

    if not wall_meshes:
        raise ValueError("build_glb: geometry['walls'] produced no valid wall meshes")

    # Overall bounding box across all walls (world X/Z), before centering.
    all_bounds = np.array([m.bounds for m in wall_meshes])  # (n, 2, 3)
    min_xyz = all_bounds[:, 0, :].min(axis=0)
    max_xyz = all_bounds[:, 1, :].max(axis=0)
    center_x = (min_xyz[0] + max_xyz[0]) / 2.0
    center_z = (min_xyz[2] + max_xyz[2]) / 2.0

    recenter = np.eye(4)
    recenter[0, 3] = -center_x
    recenter[2, 3] = -center_z
    for m in wall_meshes:
        m.apply_transform(recenter)

    # Floor slab: 0.02 m thick, spanning the (recentered) wall footprint with a small margin.
    floor_thickness = 0.02
    margin = 0.1
    floor_w = (max_xyz[0] - min_xyz[0]) + 2 * margin
    floor_d = (max_xyz[2] - min_xyz[2]) + 2 * margin
    floor = trimesh.creation.box(extents=[max(floor_w, 0.01), floor_thickness, max(floor_d, 0.01)])
    floor.apply_translation([0.0, -floor_thickness / 2.0, 0.0])
    floor.visual = trimesh.visual.texture.TextureVisuals(
        material=trimesh.visual.material.PBRMaterial(
            baseColorFactor=[90, 90, 95, 255],
            metallicFactor=0.0,
            roughnessFactor=1.0,
        )
    )
    floor.metadata["name"] = "floor"

    scene = trimesh.Scene()
    for name, mesh in zip(names, wall_meshes):
        scene.add_geometry(mesh, node_name=name, geom_name=name)
    scene.add_geometry(floor, node_name="floor", geom_name="floor")

    scene.export(out_path, file_type="glb")

    total_vertices = sum(len(g.vertices) for g in scene.geometry.values())
    total_faces = sum(len(g.faces) for g in scene.geometry.values())
    bmin, bmax = scene.bounds
    print(f"Exported: {out_path}")
    print(f"Nodes: {len(wall_meshes)} walls + 1 floor")
    print(f"Vertices: {total_vertices}  Faces: {total_faces}")
    print(f"Bounding box (m): min={bmin.round(3).tolist()} max={bmax.round(3).tolist()} "
          f"size={(bmax - bmin).round(3).tolist()}")

    return scene


In [ ]:
# Cell 11 — Stage A export: GLB for Scene Viewer
# Provisional scale for the AR proof: CubiCasa RATIO 0.01 m/px -> 100 px/m.
# The real user-supplied scale is applied at Unity placement (Stage B); this only
# sizes the Stage A walkthrough sensibly.
if report["accepted"]:
    PPM = 100.0
    geo = json.load(open("/kaggle/working/sample_response.json"))
    for w in geo["walls"]:
        w["thickness"] = float(w.get("thickness", t_med)) / PPM     # export is px; build_glb wants meters
    build_glb(geo, pixels_per_meter=PPM, wall_height_m=WALL_HEIGHT)
else:
    print("rejected — no GLB")